# Powered landing after a free fall (cold-gas N2 thruster, regulated chamber pressure)

The vehicle is released from rest at height `h0` (25 m) and falls freely. At the **ignition altitude** `z_ign` the regulated thruster opens. With *downward positive* velocity `v`:

$$\frac{dv}{dt}=a = g-\frac{F+D}{M(t)},\qquad \frac{dz}{dt}=-v,\qquad \frac{dM}{dt}=-\dot m$$

* $a=0$ is the **balance point**: thrust equals weight ($F=Mg$). This is the instant of maximum speed; afterwards the vehicle decelerates ($a<0$). With an on/off valve and $T/W>1$ at ignition, balance happens right at ignition; with a throttle ramp it would come later.
* The notebook **solves for the ignition altitude** that gives **v = 0 exactly at touchdown**, then reports the balance time, touchdown thrust margin, gas used and peak deceleration.
* Chamber pressure is `min(P_reg, tank pressure)`: constant thrust while the regulator can hold `P_reg`, blow-down after that.
* For a choked plain converging nozzle, $F \approx (1.268\,P_c - P_{atm})\,A_t$ (e.g. 33.0 N at $P_c$ = 10 bar, 6 mm throat). The ambient-pressure term matters at these pressures.

**Important:** this is an *open-loop* (fixed ignition altitude, fixed thrust) landing. Cell 5b shows how fragile that is. A real vehicle needs an altimeter and a throttled or pulsed valve.

Run the cells top to bottom. Change inputs in **Cell 1** only.

In [12]:
# ============ CELL 1 : INPUTS ============
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import brentq

# ---- gas (Nitrogen) and environment ----
gamma = 1.4;  R = 296.8;  g0 = 9.80665;  P_atm = 1.01325e5;  rho_air = 1.2
T_s1  = 300.0          # K   initial gas temperature in tank

# ---- drop ----
h0    = 20.0           # m   release height of the centre of gravity
CdA   = 0.0            # m^2 drag area (Cd*A). 0 = no drag.  e.g. 0.05 for a small compact body

# ---- storage tank + vehicle mass ----
P_s1      = 20e5       # Pa   initial storage pressure
D_in, L_cyl = 0.0508, 1   # m  tank internal diameter, cylinder length
material  = "Al6061-T6"    # "Al6061-T6", "Steel 4130", "SS304", "COPV (effective)"
SF, t_min, fit_factor = 2.5, 1.0e-3, 1.25
m_other   = 0.30       # kg  valve + regulator0+ plumbing + nozzle + electronics + payload
M_total_override = None    # kg  set a LIFT-OFF mass (gas included) to ignore the tank model

# ---- nozzle + regulator ----
d_throat  = 8.0e-3     # m   throat diameter
eps       = 1.0        # exit/throat area ratio (1 = plain converging nozzle)
P_reg     = 5e5       # Pa  regulated chamber pressure (set None for pure blow-down)

# ---- real-hardware effects (ideal values by default: Cd = 1, no valve lag) ----
Cd          = 1.0      # -   discharge/thrust efficiency (real nozzles + valve + regulator: ~0.90-0.97)
valve_delay = 0.0      # s   time from 'open' command until flow starts (solenoids: ~0.01-0.10 s)
valve_ramp  = 0.0      # s   time to go from closed to fully open
reg_dropout = 0.0      # Pa  regulator needs tank pressure >= P_reg + reg_dropout to hold P_reg (typ. 2e5 - 5e5)
leg_len     = 0.0      # m   height of the centre of gravity above the ground when the feet touch (h0 is the CG height at release)

dt, t_max = 1.0e-3, 20.0

In [13]:
# ============ CELL 2 : TANK, MASS, NOZZLE MODEL (same as the hopper notebook) ============
MATERIALS = {"Al6061-T6": (290e6, 2700.0), "Steel 4130": (670e6, 7850.0),
             "SS304": (515e6, 8000.0), "COPV (effective)": (1500e6, 1700.0)}

def tank_props(D_in, L_cyl, P_s1, material, SF=None, t_min=None, fit_factor=None):
    SF = globals()['SF'] if SF is None else SF; t_min = globals()['t_min'] if t_min is None else t_min
    fit_factor = globals()['fit_factor'] if fit_factor is None else fit_factor
    sig, rho = MATERIALS[material]; Ri = D_in/2
    V = np.pi*Ri**2*L_cyl + 4/3*np.pi*Ri**3
    t_cyl, t_head = max(SF*P_s1*Ri/sig, t_min), max(SF*P_s1*Ri/(2*sig), t_min)
    return V, fit_factor*rho*(2*np.pi*Ri*L_cyl*t_cyl + 4*np.pi*Ri**2*t_head), t_cyl

K_choke = (2/(gamma+1))**((gamma+1)/(2*(gamma-1)))
PR_crit = ((gamma+1)/2)**(gamma/(gamma-1))
def area_ratio(M): return (1/M)*((2/(gamma+1))*(1+(gamma-1)/2*M**2))**((gamma+1)/(2*(gamma-1)))
def mach_from_eps(e): return 1.0 if e <= 1.0001 else brentq(lambda M: area_ratio(M)-e, 1.0000001, 12)
def mach_from_pr(pr): return np.sqrt(2/(gamma-1)*(pr**((gamma-1)/gamma)-1))

def thrust_mdot(Pc, Tc, At, eps, Me_design, sep_limit=0.4):
    # (F, mdot, Ve, Pe).  Choked: fixed-area-ratio ideal expansion (separation-limited). Unchoked: book Stage II.
    if Pc <= P_atm: return 0.0, 0.0, 0.0, P_atm
    if Pc >= PR_crit*P_atm:
        mdot = Pc*At*np.sqrt(gamma/(R*Tc))*K_choke
        Me, Ae = Me_design, At*eps
        Pe = Pc*(1+(gamma-1)/2*Me**2)**(-gamma/(gamma-1))
        if Pe < sep_limit*P_atm:
            Pe = sep_limit*P_atm; Me = mach_from_pr(Pc/Pe); Ae = At*area_ratio(Me)
        Te = Tc/(1+(gamma-1)/2*Me**2); Ve = Me*np.sqrt(gamma*R*Te)
        return mdot*Ve + (Pe-P_atm)*Ae, mdot, Ve, Pe
    Me = min(mach_from_pr(Pc/P_atm), 1.0)
    Te = Tc/(1+(gamma-1)/2*Me**2); Ve = Me*np.sqrt(gamma*R*Te)
    mdot = P_atm/(R*Te)*At*Ve
    return mdot*Ve, mdot, Ve, P_atm

V, m_tank, _ = tank_props(D_in, L_cyl, P_s1, material)
m_gas0 = P_s1/(R*T_s1)*V
if M_total_override is None:
    M_dry = m_tank + m_other
else:                                          # keep the gas mass from the tank model, adjust the dry mass
    M_dry = M_total_override - m_gas0
M0 = M_dry + m_gas0
print(f"Tank {V*1e3:.2f} L | gas {m_gas0*1e3:.0f} g | dry {M_dry*1e3:.0f} g | lift-off mass {M0*1e3:.0f} g (W = {M0*g0:.2f} N)")
print(f"Free-fall impact speed without thrust: {np.sqrt(2*g0*h0):.1f} m/s after {np.sqrt(2*h0/g0):.2f} s")
F_start, *_ = thrust_mdot(min(P_reg or P_s1, P_s1), T_s1, np.pi/4*d_throat**2, eps, mach_from_eps(eps))
print(f"Thrust at ignition: {F_start:.1f} N  ->  T/W = {F_start/(M0*g0):.2f}")

Tank 2.10 L | gas 47 g | dry 866 g | lift-off mass 913 g (W = 8.95 N)
Free-fall impact speed without thrust: 19.8 m/s after 2.02 s
Thrust at ignition: 26.8 N  ->  T/W = 2.99


In [3]:
# ============ CELL 3 : DROP SIMULATION + IGNITION-ALTITUDE SOLVER ============
_U = object()     # sentinel: "use the CURRENT Cell-1 value" (so editing Cell 1 can never be silently ignored)
def drop(z_ign, d_throat=_U, P_reg=_U, eps=_U, M_dry=_U, h0=_U, dt=_U, record=True, P_s1=_U, V=_U,
         Cd=_U, valve_delay=_U, valve_ramp=_U):
    G = globals()
    d_throat = G["d_throat"] if d_throat is _U else d_throat
    P_reg    = G["P_reg"]    if P_reg    is _U else P_reg
    eps      = G["eps"]      if eps      is _U else eps
    M_dry    = G["M_dry"]    if M_dry    is _U else M_dry
    h0       = G["h0"]       if h0       is _U else h0
    dt       = G["dt"]       if dt       is _U else dt
    P_s1     = G["P_s1"]     if P_s1     is _U else P_s1
    Cd, valve_delay, valve_ramp = [G[k] if v is _U else v for k, v in
                                   (("Cd", Cd), ("valve_delay", valve_delay), ("valve_ramp", valve_ramp))]
    V_ = tank_props(D_in, L_cyl, P_s1, material)[0] if V is _U else V
    m1 = P_s1/(R*T_s1)*V_
    Md = M_dry
    At = np.pi/4*d_throat**2; Me_d = mach_from_eps(eps)
    z, v, t, m = h0, 0.0, 0.0, m1
    ignited = False; t_ign = None; bal = None
    rec = {k: [] for k in ["t","z","v","a","F","W","Pc","mdot"]}
    status = "running"
    # --- free fall down to the ignition altitude, solved EXACTLY (with quadratic drag), so that the ignition point is not
    #     quantised to the time step: this keeps miss(z_ign) continuous and the ignition solver reliable ---
    if G["leg_len"] < z_ign < h0:
        M_pre = Md + m1; s_fall = h0 - z_ign; CdA_ = G["CdA"]
        if CdA_ <= 0:
            t_f = np.sqrt(2*s_fall/g0); fz = lambda tt: (h0 - 0.5*g0*tt**2, g0*tt, g0)
        else:
            vt = np.sqrt(2*M_pre*g0/(rho_air*CdA_))                       # terminal speed
            t_f = vt/g0*np.arccosh(np.exp(s_fall*g0/vt**2))
            fz = lambda tt: (h0 - vt**2/g0*np.log(np.cosh(g0*tt/vt)), vt*np.tanh(g0*tt/vt), g0/np.cosh(g0*tt/vt)**2)
        if record:
            for tt in np.arange(0.0, t_f, dt):
                zz, vv, aa = fz(tt)
                for k, val in zip(rec, [tt, zz, vv, aa, 0.0, M_pre*g0, 0.0, 0.0]): rec[k].append(val)
        z, v, t = z_ign, fz(t_f)[1], t_f
        ignited, t_ign = True, t_f
    while t < t_max:
        ratio = m/m1; Ps = P_s1*ratio**gamma; Ts = T_s1*ratio**(gamma-1)
        if not ignited and z <= z_ign: ignited, t_ign = True, t
        if ignited and m > 0:
            Pc = Ps if P_reg is None else max(min(P_reg, Ps - G['reg_dropout']), 0.0)
            F, mdot, Ve, _ = thrust_mdot(Pc, Ts, At, eps, Me_d)
            tau = t - t_ign - valve_delay                      # valve lag and opening ramp
            frac = 0.0 if tau < 0 else (1.0 if valve_ramp <= 0 or tau >= valve_ramp else tau/valve_ramp)
            F, mdot = F*Cd*frac, mdot*Cd*frac
        else:
            Pc, F, mdot = 0.0, 0.0, 0.0
        M = Md + m; W = M*g0
        D = 0.5*rho_air*CdA*v*abs(v)          # always opposes the motion
        a = g0 - (F + D)/M
        if bal is None and ignited and F >= W: bal = dict(t=t, z=z, v=v)
        if record:
            for k, val in zip(rec, [t, z, v, a, F, W, Pc, mdot]): rec[k].append(val)
        v += a*dt; z -= v*dt; m = max(m - mdot*dt, 0.0); t += dt
        if z <= G["leg_len"]: status = "touchdown"; z = G["leg_len"]; break
        if ignited and v <= 0.0 and t > t_ign + valve_delay + valve_ramp + 2*dt: status = "stopped_in_air"; break
    out = dict(status=status, t_end=t, z_end=z - G['leg_len'], v_end=v, t_ign=t_ign, bal=bal, gas_used=m1-m, m_gas=m1,
               M_end=Md+m, F_end=F, W_end=(Md+m)*g0)
    if record:
        out["rec"] = {k: np.array(x) for k, x in rec.items()}
        out["peak_decel_g"] = max(0.0, -rec["a"][int(np.argmin(rec["a"]))]/g0) if rec["a"] else 0.0   # net kinematic deceleration
        out["peak_load_g"]  = float(np.max(np.array(rec["F"])/np.array(rec["W"])))                        # felt load = thrust / weight
    return out

def miss(z_ign, **kw):
    # >0: touches down while still moving (ignited too late);  <0: stops in the air (ignited too early)
    s = drop(z_ign, record=False, **kw)
    return s["v_end"] if s["status"] == "touchdown" else -s["z_end"]

def _why_not(z, kw):
    # explain a failed solve from the best attempt at ignition altitude z
    s_ = drop(z, record=False, **kw)
    left = (s_["m_gas"] - s_["gas_used"])*1e3
    cause = ("the gas ran out" if left < 0.01*s_["m_gas"]*1e3 else
             "thrust fell below weight while gas remained (T/W < 1 at ignition, or the tank dropped below the regulator setting and the flow went into blow-down)")
    return (f"cannot stop: best attempt reached the ground at {abs(s_['v_end']):.1f} m/s with thrust {s_['F_end']:.1f} N vs weight {s_['W_end']:.1f} N "
            f"and {left:.0f} g gas left -> {cause}")

def solve_ignition(**kw):
    hi = kw.get("h0", h0) - 0.05
    if miss(0.0, **kw) <= 0: return 0.0, "no burn needed"
    if miss(hi, **kw) > 0:   return None, _why_not(hi, kw)
    z = brentq(lambda z: miss(z, **kw), 0.0, hi, xtol=1e-4)
    # Ignition is solved exactly, so miss() is continuous at a true root: across a +-2 mm window it changes by about
    # slope*4 mm (slope measured just below the root). A regime jump (thrust decayed below weight / gas out) changes
    # it by metres or m/s whatever the slope.
    z_far = max(z - 0.05, 0.0)
    slope = abs(miss(z_far, **kw) - miss(z - 2e-3, **kw))/max(z - 2e-3 - z_far, 1e-3)
    if abs(miss(z + 2e-3, **kw) - miss(z - 2e-3, **kw)) > 3*slope*4e-3 + 0.3:
        return None, _why_not(z - 2e-3, kw)
    return z, "ok"

z_ign, msg = solve_ignition(M_dry=M_dry)
print("ignition altitude solver:", msg, "" if z_ign is None else f"-> z_ign = {z_ign:.3f} m")

ignition altitude solver: ok -> z_ign = 7.448 m


In [4]:
# ============ CELL 4 : RESULTS ============
if z_ign is None:
    s = drop(0.0, M_dry=M_dry); r = s["rec"]
    print(f"NO SOFT LANDING with these inputs: {msg}.  (T/W at ignition = {F_start/(M0*g0):.2f}, gas = {m_gas0*1e3:.0f} g)")
    print("Try a larger d_throat or P_reg (more thrust = shorter burn = less gas), more gas, or a lighter vehicle.")
else:
    s = drop(z_ign, M_dry=M_dry); r = s["rec"]
    b = s["bal"]
    i_ign = int(np.argmin(abs(r["t"] - s["t_ign"])))             # speed taken from the simulation (includes drag)
    print(f"Ignition command: z = {z_ign:.2f} m, t = {s['t_ign']:.3f} s after release, simulated speed {r['v'][i_ign]:.1f} m/s")
    if b: print(f"Thrust reaches weight (a = 0 = maximum speed {b['v']:.1f} m/s) at t = {b['t']:.3f} s, z = {b['z']:.2f} m"
                + ("  -> effectively at ignition: on/off valve with T/W > 1" if b['t'] - s['t_ign'] < 0.02 + valve_delay + valve_ramp else ""))
    if s["status"] == "touchdown":
        print(f"Touchdown: t = {s['t_end']:.3f} s, speed = {s['v_end']:.2f} m/s")
    else:   # solver puts the stop (v = 0) a few mm above the ground = soft touchdown; close the valve here
        print(f"Soft touchdown (v = 0) at t = {s['t_end']:.3f} s, {s['z_end']*100:.1f} cm above the ground")
    print(f"Thrust still available at touchdown: {s['F_end']:.1f} N vs weight {s['W_end']:.1f} N  -> T/W = {s['F_end']/s['W_end']:.2f} (margin, close the valve there)")
    print(f"Gas left at touchdown: {(s['m_gas']-s['gas_used'])*1e3:.0f} g")
    print(f"Peak net deceleration: {s['peak_decel_g']:.2f} g | peak FELT load (thrust/weight, what the structure carries): {s['peak_load_g']:.2f} g | burn time {s['t_end']-s['t_ign']:.2f} s")
    print(f"Gas used: {s['gas_used']*1e3:.1f} g of {s['m_gas']*1e3:.0f} g  ({s['gas_used']/s['m_gas']*100:.0f} %)")

Ignition command: z = 7.45 m, t = 1.600 s after release, simulated speed 15.7 m/s
Thrust reaches weight (a = 0 = maximum speed 15.7 m/s) at t = 1.600 s, z = 7.45 m  -> effectively at ignition: on/off valve with T/W > 1
Soft touchdown (v = 0) at t = 2.537 s, 0.0 cm above the ground
Thrust still available at touchdown: 26.8 N vs weight 9.6 N  -> T/W = 2.79 (margin, close the valve there)
Gas left at touchdown: 42 g
Peak net deceleration: 1.79 g | peak FELT load (thrust/weight, what the structure carries): 2.79 g | burn time 0.94 s
Gas used: 58.2 g of 100 g  (58 %)


In [5]:
# ============ CELL 5 : PLOTS ============
fig, ax = plt.subplots(2, 3, figsize=(15, 8))
t = r["t"]
ax[0,0].plot(t, r["z"]);                 ax[0,0].set_title("Altitude [m]")
ax[0,1].plot(t, r["v"]);                 ax[0,1].set_title("Speed, downward [m/s]"); ax[0,1].axhline(0, color="k", lw=.5)
ax[0,2].plot(t, r["a"]/g0);              ax[0,2].set_title("a = g - T/m  [in g]"); ax[0,2].axhline(0, color="r", ls="--")
ax[1,0].plot(t, r["F"], label=24"thrust"); ax[1,0].plot(t, r["W"], "r--", label="weight"); ax[1,0].set_title("Thrust vs weight [N]"); ax[1,0].legend()
ax[1,1].plot(t, r["Pc"]/1e5);            ax[1,1].set_title("Chamber pressure [bar]")
ax[1,2].plot(t, r["mdot"]*1e3);          ax[1,2].set_title("Mass flow [g/s]")
for a_ in ax.ravel():
    a_.grid(alpha=.3); a_.set_xlabel("t [s]")
    if z_ign is not None and s["bal"]: a_.axvline(s["bal"]["t"], color="g", ls=":", lw=1)
plt.suptitle("Green dotted line = balance point (thrust = weight)"); plt.tight_layout(); plt.show()

SyntaxError: invalid syntax. Perhaps you forgot a comma? (4162026632.py, line 7)

In [ ]:
# ============ CELL 5b : HOW FRAGILE IS THIS OPEN-LOOP LANDING? ============
# Ignition altitude is fixed at the 'perfect' value above; then ONE thing at a time is perturbed.
if z_ign is None:
    print("no nominal landing solution - nothing to perturb")
else:
    def touch(dz=0.0, mass_k=1.0, thrust_k=1.0, **kw):
        return drop(z_ign + dz, M_dry=M_dry*mass_k, Cd=Cd*thrust_k, record=False, dt=1e-3, **kw)
    rows = []
    def add(label, s_):
        if s_["status"] == "touchdown":
            res, v_hit = f"hits the ground at {s_['v_end']:.1f} m/s", s_["v_end"]
        else:
            v_hit = np.sqrt(2*g0*s_["z_end"])
            res = f"stops {s_['z_end']*100:.0f} cm up (then falls: {v_hit:.1f} m/s on impact if valve is closed)"
        rows.append({"case": label, "outcome": res, "impact speed [m/s]": round(v_hit, 1),
                     "equivalent drop height [cm]": round(v_hit**2/(2*g0)*100)})
    for dz in (-0.5, -0.25, -0.1, 0.1, 0.25, 0.5): add(f"ignite {abs(dz)} m {'lower' if dz < 0 else 'higher'}", touch(dz=dz))
    for dl in (0.02, 0.05, 0.10):                  add(f"valve opens {dl*1e3:.0f} ms late", touch(valve_delay=dl))
    for k in (0.90, 0.95, 1.05, 1.10):             add(f"thrust and flow x{k}", touch(thrust_k=k))
    for k in (0.9, 1.1):                           add(f"vehicle dry mass {k-1:+.0%}", touch(mass_k=k))
    display(pd.DataFrame(rows))
    v_ign = np.sqrt(2*g0*(h0 - z_ign))
    print(f"At ignition the vehicle falls {v_ign:.1f} m/s, so 10 ms of timing error = {v_ign*0.01*100:.0f} cm of altitude.")

In [ ]:
# ============ CELL 6 : HOW MUCH THRUST?  sweep the throat diameter at the chosen P_reg ============
rows = []
for d in np.linspace(2.0e-3, 9e-3, 15):
    F0, *_ = thrust_mdot(min(P_reg or P_s1, P_s1), T_s1, np.pi/4*d**2, eps, mach_from_eps(eps))
    zi, msg = solve_ignition(M_dry=M_dry, d_throat=d, dt=2e-3)
    if zi is None:
        rows.append(dict(d_mm=d*1e3, F_N=F0, TW=F0/(M0*g0), z_ign_m=np.nan, gas_g=np.nan, note="no solution")); continue
    s_ = drop(zi, M_dry=M_dry, d_throat=d, dt=2e-3)
    rows.append(dict(d_mm=d*1e3, F_N=F0, TW=F0/(M0*g0), z_ign_m=zi, gas_g=s_["gas_used"]*1e3,
                     net_decel_g=s_["peak_decel_g"], felt_load_g=s_["peak_load_g"], burn_s=s_["t_end"]-s_["t_ign"], note="ok"))
sw = pd.DataFrame(rows)
display(sw.round(2))
ok = sw[sw.note == "ok"]   # rows marked "no solution": too little thrust, or the gas runs out before the stop
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(ok.TW, ok.z_ign_m, "o-");  ax[0].set_title("Ignition altitude [m]")
ax[1].plot(ok.TW, ok.gas_g, "o-");    ax[1].set_title("Gas used [g]")
ax[2].plot(ok.TW, ok.felt_load_g, "o-");   ax[2].set_title("Peak felt load, thrust/weight [g]")
for a_ in ax: a_.set_xlabel("thrust / weight at ignition"); a_.grid(alpha=.3)
plt.tight_layout(); plt.show()

In [ ]:
# ============ CELL 7 : FEASIBILITY MAP  regulator pressure x throat diameter (tank as in Cell 1) ============
# blank = no soft landing possible (too little thrust, or the gas runs out before the vehicle stops)
Pr_list = np.array([6, 8, 10, 12, 15, 18])*1e5
d_list  = np.array([4, 5, 6, 7, 8, 9])*1e-3
Zi = pd.DataFrame(index=[f"{p/1e5:.0f} bar" for p in Pr_list], columns=[f"{d*1e3:.0f} mm" for d in d_list], dtype=float)
Gu, Pk = Zi.copy(), Zi.copy()
for p in Pr_list:
    for d in d_list:
        zi, msg = solve_ignition(M_dry=M_dry, d_throat=d, P_reg=min(p, P_s1), dt=3e-3)
        if zi is None: continue
        s_ = drop(zi, M_dry=M_dry, d_throat=d, P_reg=min(p, P_s1), dt=3e-3)
        Zi.loc[f"{p/1e5:.0f} bar", f"{d*1e3:.0f} mm"] = zi
        Gu.loc[f"{p/1e5:.0f} bar", f"{d*1e3:.0f} mm"] = s_["gas_used"]*1e3
        Pk.loc[f"{p/1e5:.0f} bar", f"{d*1e3:.0f} mm"] = s_["peak_load_g"]
print("Ignition altitude [m]  (rows: regulator pressure, columns: throat diameter)"); display(Zi.round(1))
print("Gas used [g]");           display(Gu.round(0))
print("Peak felt load, thrust/weight [g]  (net deceleration = this minus 1)");  display(Pk.round(1))

In [ ]:
# ============ CELL 8 : STORAGE PRESSURE  (tank mass is recomputed for each pressure) ============
# The regulator only holds P_reg while tank pressure > P_reg, so the gas available at full thrust is
# about  m_gas*(1 - (P_reg/P_s)^(1/gamma)).  A higher storage pressure buys a longer regulated burn.
rows = []
for Ps in np.array([20, 30, 40, 60, 100])*1e5:
    V_, mt, _ = tank_props(D_in, L_cyl, Ps, material)
    Md = mt + m_other; mg = Ps/(R*T_s1)*V_
    for d in [4e-3, 5e-3, 6e-3, 7e-3]:
        zi, msg = solve_ignition(M_dry=Md, d_throat=d, P_s1=Ps, P_reg=(None if P_reg is None else min(P_reg, Ps)), dt=3e-3)
        if zi is None:
            rows.append(dict(P_storage_bar=Ps/1e5, throat_mm=d*1e3, M0_g=(Md+mg)*1e3, gas_g=mg*1e3, note="no solution")); continue
        s_ = drop(zi, M_dry=Md, d_throat=d, P_s1=Ps, P_reg=(None if P_reg is None else min(P_reg, Ps)), dt=3e-3)
        rows.append(dict(P_storage_bar=Ps/1e5, throat_mm=d*1e3, M0_g=(Md+mg)*1e3, gas_g=mg*1e3, z_ign_m=zi,
                         gas_used_g=s_["gas_used"]*1e3, net_decel_g=s_["peak_decel_g"], felt_load_g=s_["peak_load_g"], note="ok"))
display(pd.DataFrame(rows).round(2))